In [36]:
import pandas as pd
import numpy as np
#merging meteostat datasets
df_1 = pd.read_csv("daily_weather_data/alexanderplatz_daily_2026.csv")
df_2 = pd.read_csv("daily_weather_data/dahlem_daily_2026.csv")
df_3 = pd.read_csv("daily_weather_data/marzahn_daily_2026.csv")
df_4 = pd.read_csv("daily_weather_data/tegel_daily_2026.csv")
df_5 = pd.read_csv("daily_weather_data/tempelhof_daily_2026.csv")

In [8]:
#rename stations
df_1["station"] = "Alexanderplatz"
df_2["station"] = "Dahlem"
df_3["station"] = "Marzahn"
df_4["station"] = "Tegel"
df_5["station"] = "Tempelhof"

In [107]:
# Assign station names and source identifiers to each dataset
meteostat_stations = [
    df_1.assign(station="Alexanderplatz", source="Meteostat"),
    df_2.assign(station="Dahlem", source="Meteostat"),
    df_3.assign(station="Marzahn", source="Meteostat"),
    df_4.assign(station="Tegel", source="Meteostat"),
    df_5.assign(station="Tempelhof", source="Meteostat")
]

# Combine all five Meteostat stations
df_weather = pd.concat(
    meteostat_stations,
    ignore_index=True
)

# Check station distribution
print(df_meteostat["station"].value_counts())

station
Alexanderplatz    303
Dahlem            303
Marzahn           303
Tegel             303
Tempelhof         303
Name: count, dtype: int64


In [6]:
#remove not needed columns
df_weather = df_weather.drop(
    columns=["snow", "wdir", "wpgt", "tsun"]
)


In [ ]:
# Check the combined dataset
print(df_weather.shape)
print(df_weather.head())
print(df_weather.info())

# Check how many observations each station contributes
print(df_weather["station"].value_counts())

# Check missing values
print(df_weather.isna().sum())

In [ ]:
df_weather = df_weather.sort_values(
    by=["date", "station"]
).reset_index(drop=True)

df_weather.head(10)

In [ ]:
#renaming columns
df_weather = df_weather.rename(columns={
    "tavg": "avg_temp",
    "tmin": "min_temp",
    "tmax": "max_temp",
    "prcp": "precipitation",
    "wspd": "wind_speed",
    "pres": "avg_air_pressure_hPa"
})

df_weather.head(10)

In [58]:
df_weather = df_weather.replace(-999, np.nan)

date     0
tavg     8
tmin     8
tmax     8
prcp     8
wspd    10
pres    10
dtype: int64

In [59]:
#saving dataset
df_weather.to_csv("berlin_weather_meteostat_combined.csv", index=False)

In [31]:
#mergind DWD datasets:
df_dwd_1 = pd.read_csv("daily_weather_data/berlin_brandenburg_daily_2026.txt", sep=";")
df_dwd_2 = pd.read_csv("daily_weather_data/buch_daily_2026.txt", sep=";")
df_dwd_3 = pd.read_csv("daily_weather_data/dahlem_daily_2026.txt", sep=";")
df_dwd_4 = pd.read_csv("daily_weather_data/marzahn_daily_2026.txt", sep=";")
df_dwd_5 = pd.read_csv("daily_weather_data/tempelhof_daily_2026.txt", sep=";")

In [32]:
df_dwd_1["station"] = "Berlin Brandenburg"
df_dwd_2["station"] = "Buch"
df_dwd_3["station"] = "Dahlem"
df_dwd_4["station"] = "Marzahn"
df_dwd_5["station"] = "Tempelhof"

In [33]:
df_dwd_weather = pd.concat(
    [df_dwd_1, df_dwd_2, df_dwd_3, df_dwd_4, df_dwd_5],
    ignore_index=True
)

In [42]:
#removing unnecessary spaces to remove columns
df_dwd_weather.columns = df_dwd_weather.columns.str.strip()

In [ ]:
#removing unnecessary columns
df_dwd_weather = df_dwd_weather.drop(
    columns=["QN_3", "FX", "FM", "QN_4", "RSKF", "SDK", "SHK_TAG", "VPM", "PM", "TGK", "eor"]
)

In [ ]:
# Convert DWD date column to datetime
df_dwd_weather["MESS_DATUM"] = pd.to_datetime(
    df_dwd_weather["MESS_DATUM"],
    format="%Y%m%d"
)

# Rename the date column
df_dwd_weather = df_dwd_weather.rename(
    columns={"MESS_DATUM": "date"}
)

In [53]:
df_dwd_weather = df_dwd_weather.rename(columns={
    "STATIONS_ID": "station_id",
    "RSK": "precipitation_mm",
    "NM": "cloud_cover_mean",
    "TMK": "daily_temp_mean",
    "UPM": "humidity_level_mean",
    "TXK": "daily_max_temp",
    "TNK": "daily_min_temp",
    "station": "station_name"
})

In [51]:
df_dwd_weather = df_dwd_weather.replace(-999, np.nan)

In [55]:
df_dwd_weather.isna().sum()

station_id                0
date                      0
precipitation_mm          8
cloud_cover_mean       1120
daily_temp_mean           7
humidity_level_mean      19
daily_max_temp            8
daily_min_temp            8
station_name              0
dtype: int64

In [61]:
df_dwd_weather.to_csv("berlin_weather_dwd_combined.csv", index=False)

In [ ]:
# Import the measurement records, skipping metadata and header rows
df_w_1 = pd.read_csv(
    "daily_weather_data/individual_datasets/flughafensee_B920_water_quality.csv",
    sep=";",
    encoding="cp1252", #added to correctly visualize umlaut
    skiprows=12, #skipping the first 11 rows
    header=None,
    dtype=str,
    engine="python"
)

# Assign the correct column names
df_w_1.columns = [
    "date",
    "bathing_water_quality",
    "coliform_bacteria",
    "cyanobacteria_chl_a",
    "e_coli",
    "intestinal_enterococci",
    "water_visibility_depth",
    "water_temperature"
]

# Display the first rows
print(df_w_1.head(10).to_string())

In [79]:
def load_water_quality(file_path):

    # Import measurement records, skipping metadata and header rows
    df = pd.read_csv(
        file_path,
        sep=";",
        encoding="cp1252",
        skiprows=12,
        header=None,
        dtype=str,
        engine="python"
    )

    # Assign consistent English column names
    df.columns = [
        "date",
        "bathing_water_quality",
        "coliform_bacteria",
        "cyanobacteria_chl_a",
        "e_coli",
        "intestinal_enterococci",
        "water_visibility_depth",
        "water_temperature"
    ]

    return df

In [81]:
df_w_2 = load_water_quality(
    "daily_weather_data/individual_datasets/grosser_wannsee_B350_water_quality.csv"
)
df_w_3 = load_water_quality(
    "daily_weather_data/individual_datasets/muggelsee_B116_water_quality.csv"
)
df_w_4 = load_water_quality(
    "daily_weather_data/individual_datasets/orankesee_B911_water_quality.csv"
)
df_w_5 = load_water_quality(
    "daily_weather_data/individual_datasets/tegeler_see_B309_water_quality.csv"
)
df_w_6 = load_water_quality(
    "daily_weather_data/individual_datasets/weissensee_B910_water_quality.csv"
)

In [82]:
water_datasets = {
    "Flughafensee": df_w_1,
    "Wannsee": df_w_2,
    "Müggelsee": df_w_3,
    "Orankesee": df_w_4,
    "Tegeler See": df_w_5,
    "Weißensee": df_w_6
}

# Check the number of rows and columns in each dataset
for lake, df in water_datasets.items():
    print(f"{lake}: {df.shape[0]} rows, {df.shape[1]} columns")

Flughafensee: 19 rows, 8 columns
Wannsee: 19 rows, 8 columns
Müggelsee: 20 rows, 8 columns
Orankesee: 20 rows, 8 columns
Tegeler See: 20 rows, 8 columns
Weißensee: 20 rows, 8 columns


In [78]:
for col in df_w_1.columns:
    null_values = df_w_1[
        df_w_1[col].str.contains("null", case=False, na=False)
    ][col].unique()

    if len(null_values) > 0:
        print(f"\n{col}:")
        print(null_values)


coliform_bacteria:
<ArrowStringArray>
['null 910', 'null 360']
Length: 2, dtype: str

cyanobacteria_chl_a:
<ArrowStringArray>
['null 0,30', 'null 0,0000']
Length: 2, dtype: str

e_coli:
<ArrowStringArray>
['null 30,00', 'null 15,00', 'null 143']
Length: 3, dtype: str

intestinal_enterococci:
<ArrowStringArray>
['null 15,00', 'null 109']
Length: 2, dtype: str

water_temperature:
<ArrowStringArray>
['null 17,80', 'null 21,50', 'null 21,20', 'null 20,60', 'null 23,10']
Length: 5, dtype: str


In [ ]:
# Store the six lake datasets together
water_datasets = {
    "Flughafensee": df_w_1,
    "Wannsee": df_w_2,
    "Müggelsee": df_w_3,
    "Orankesee": df_w_4,
    "Tegeler See": df_w_5,
    "Weißensee": df_w_6
}

# Count status-prefixed values in each dataset
for lake, df in water_datasets.items():
    count = df.astype("string").apply(
        lambda col: col.str.contains(
            "null", case=False, na=False
        ).sum()
    ).sum()

    print(f"{lake}: {count} values containing 'null'")

In [83]:
# Count status-prefixed values in each dataset
for lake, df in water_datasets.items():
    count = df.astype("string").apply(
        lambda col: col.str.contains(
            "null", case=False, na=False
        ).sum()
    ).sum()

    print(f"{lake}: {count} values containing 'null'")

Flughafensee: 20 values containing 'null'
Wannsee: 19 values containing 'null'
Müggelsee: 21 values containing 'null'
Orankesee: 18 values containing 'null'
Tegeler See: 20 values containing 'null'
Weißensee: 18 values containing 'null'


In [ ]:
# Count values containing 'null' in each column and lake
for lake, df in water_datasets.items():
    print(f"\n--- {lake} ---")

    for col in df.columns:
        count = df[col].str.contains(
            "null", case=False, na=False
        ).sum()

        if count > 0:
            print(f"{col}: {count}")

In [85]:
# Collect records containing 'null' across all lakes
null_records = []

for lake, df in water_datasets.items():

    # Identify rows containing 'null' in any column
    mask = df.astype("string").apply(
        lambda col: col.str.contains(
            "null", case=False, na=False
        )
    ).any(axis=1)

    # Keep the affected rows and identify their lake
    temp = df.loc[mask].copy()
    temp.insert(0, "lake", lake)

    null_records.append(temp)

# Combine all affected records
df_null_records = pd.concat(null_records, ignore_index=True)

# Display the result
print(df_null_records.to_string(index=False))

        lake       date bathing_water_quality coliform_bacteria cyanobacteria_chl_a     e_coli intestinal_enterococci water_visibility_depth water_temperature
Flughafensee 11.06.2025                   NaN          null 910           null 0,30 null 30,00             null 15,00                 > 1,00        null 17,80
Flughafensee 10.07.2025                   NaN             < 300         null 0,0000 null 15,00             null 15,00                 > 1,00        null 21,50
Flughafensee 24.07.2025                   NaN             < 300                 NaN   null 143               null 109                 > 1,00        null 21,20
Flughafensee 06.08.2025                   NaN             < 300         null 0,0000 null 15,00                < 15,00                 > 1,00        null 20,60
Flughafensee 20.08.2025                   NaN          null 360         null 0,0000 null 30,00                < 15,00                 > 1,00        null 23,10
Flughafensee 03.09.2025                  1,00 

In [86]:
print("DWD columns:")
print(df_dwd_weather.columns.tolist())

display(df_dwd_weather.head())

DWD columns:
['station_id', 'date', 'precipitation_mm', 'cloud_cover_mean', 'daily_temp_mean', 'humidity_level_mean', 'daily_max_temp', 'daily_min_temp', 'station_name']


,station_id,date,precipitation_mm,cloud_cover_mean,daily_temp_mean,humidity_level_mean,daily_max_temp,daily_min_temp,station_name
0,427,2025-04-09,0.0,NaN,8.9,59.0,15.4,1.8,Berlin Brandenburg
1,427,2025-04-10,0.0,5.3,9.1,66.0,14.0,5.1,Berlin Brandenburg
2,427,2025-04-11,0.0,7.2,9.7,69.0,14.8,4.9,Berlin Brandenburg
3,427,2025-04-12,0.0,4.5,11.1,74.0,17.4,4.3,Berlin Brandenburg
4,427,2025-04-13,0.0,4.5,14.8,63.0,22.8,6.8,Berlin Brandenburg


In [87]:
print("Meteostat columns:")
print(df_weather.columns.tolist())

display(df_weather.head())

Meteostat columns:
['date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres']


,date,tavg,tmin,tmax,prcp,wspd,pres
0,2026-01-01 00:00:00,2.5,1.8,3.1,3.6,23.9,998.6
1,2026-01-02 00:00:00,2.2,0.7,2.8,1.7,25.2,988.9
2,2026-01-03 00:00:00,0.0,-0.6,0.5,3.5,19.0,996.3
3,2026-01-04 00:00:00,-1.3,-2.8,-0.4,1.0,16.8,1004.3
4,2026-01-05 00:00:00,-2.0,-4.7,-0.7,0.2,11.9,1012.1


In [ ]:
#comparing Dahlem, Tempelhof and Marzahn weather stations between DWD and Meteostat datasets

# Match the Meteostat and DWD DataFrames by location
weather_pairs = {
    "Dahlem": (df_2, df_dwd_3),
    "Marzahn": (df_3, df_dwd_4),
    "Tempelhof": (df_5, df_dwd_5)
}

# Compare date coverage for each location
for location, (df_meteostat, df_dwd) in weather_pairs.items():

    # Convert Meteostat dates
    meteostat_dates = pd.to_datetime(
        df_meteostat["date"], errors="coerce"
    ).dt.date

    # Convert original DWD date format (YYYYMMDD)
    dwd_dates = pd.to_datetime(
        df_dwd["MESS_DATUM"].astype(str).str.strip(),
        format="%Y%m%d",
        errors="coerce"
    ).dt.date

    # Create sets of valid dates
    meteostat_set = set(meteostat_dates.dropna())
    dwd_set = set(dwd_dates.dropna())

    # Identify shared and source-specific dates
    shared_dates = meteostat_set.intersection(dwd_set)
    meteostat_only = meteostat_set - dwd_set
    dwd_only = dwd_set - meteostat_set

    print(f"\n{'='*50}")
    print(f"LOCATION: {location}")
    print(f"{'='*50}")

    print(f"Meteostat records: {len(df_meteostat)}")
    print(f"DWD records: {len(df_dwd)}")

    print(f"Meteostat date range: {min(meteostat_set)} to {max(meteostat_set)}")
    print(f"DWD date range: {min(dwd_set)} to {max(dwd_set)}")

    print(f"Unique Meteostat dates: {len(meteostat_set)}")
    print(f"Unique DWD dates: {len(dwd_set)}")

    print(f"Dates available in both: {len(shared_dates)}")
    print(f"Dates only in Meteostat: {len(meteostat_only)}")
    print(f"Dates only in DWD: {len(dwd_only)}")

    print(f"Invalid Meteostat dates: {meteostat_dates.isna().sum()}")
    print(f"Invalid DWD dates: {dwd_dates.isna().sum()}")

    print(f"Duplicate Meteostat dates: {meteostat_dates.duplicated().sum()}")
    print(f"Duplicate DWD dates: {dwd_dates.duplicated().sum()}")

In [91]:
# Inspect the relevant DWD columns and their data types
for location, df_dwd in [
    ("Dahlem", df_dwd_3),
    ("Marzahn", df_dwd_4),
    ("Tempelhof", df_dwd_5)
]:
    print(f"\n--- {location}: DWD ---")

    display(
        df_dwd[
            ["MESS_DATUM", " TMK", " TNK", " TXK", " RSK"]
        ].head()
    )

    print(df_dwd[[" TMK", " TNK", " TXK", " RSK"]].dtypes)

# Inspect Meteostat values and data types
for location, df_meteostat in [
    ("Dahlem", df_2),
    ("Marzahn", df_3),
    ("Tempelhof", df_5)
]:
    print(f"\n--- {location}: Meteostat ---")

    display(
        df_meteostat[
            ["date", "tavg", "tmin", "tmax", "prcp"]
        ].head()
    )

    print(df_meteostat[["tavg", "tmin", "tmax", "prcp"]].dtypes)


--- Dahlem: DWD ---


,MESS_DATUM,TMK,TNK,TXK,RSK
0,20250409,9.1,3.6,15.4,0.0
1,20250410,9.3,5.3,14.0,0.0
2,20250411,10.0,6.2,14.8,0.0
3,20250412,11.2,6.0,17.9,0.0
4,20250413,14.7,6.1,22.2,0.0


TMK    float64
TNK    float64
TXK    float64
RSK    float64
dtype: object

--- Marzahn: DWD ---


,MESS_DATUM,TMK,TNK,TXK,RSK
0,20250409,9.3,4.6,14.8,0.0
1,20250410,9.6,4.4,14.7,0.0
2,20250411,9.9,6.5,14.7,0.0
3,20250412,12.1,7.0,17.9,0.0
4,20250413,15.8,8.4,22.9,0.0


TMK    float64
TNK    float64
TXK    float64
RSK    float64
dtype: object

--- Tempelhof: DWD ---


,MESS_DATUM,TMK,TNK,TXK,RSK
0,20250409,9.4,3.4,15.6,0.0
1,20250410,9.6,5.2,14.4,0.0
2,20250411,9.9,4.6,15.3,0.0
3,20250412,11.6,4.9,17.9,0.0
4,20250413,15.6,7.8,22.7,0.0


... (3 outputs left out)

,date,tavg,tmin,tmax,prcp
0,2026-01-01 00:00:00,3.4,2.1,4.5,0.0
1,2026-01-02 00:00:00,2.1,-0.7,3.1,3.0
2,2026-01-03 00:00:00,0.0,-1.1,0.8,5.3
3,2026-01-04 00:00:00,-1.1,-3.9,0.4,0.6
4,2026-01-05 00:00:00,-1.5,-6.8,1.0,0.0


tavg    float64
tmin    float64
tmax    float64
prcp    float64
dtype: object

--- Tempelhof: Meteostat ---


,date,tavg,tmin,tmax,prcp
0,2026-01-01 00:00:00,3.4,2.2,4.5,0.0
1,2026-01-02 00:00:00,2.0,-0.5,3.2,2.0
2,2026-01-03 00:00:00,0.0,-1.0,0.8,5.0
3,2026-01-04 00:00:00,-1.3,-4.2,0.4,0.9
4,2026-01-05 00:00:00,-1.4,-9.0,0.4,0.0


tavg    float64
tmin    float64
tmax    float64
prcp    float64
dtype: object


In [92]:
# Check missing values and possible DWD missing-value codes
for location, df_dwd in [
    ("Dahlem", df_dwd_3),
    ("Marzahn", df_dwd_4),
    ("Tempelhof", df_dwd_5)
]:
    # Remove whitespace from column names in a temporary copy
    df_check = df_dwd.copy()
    df_check.columns = df_check.columns.str.strip()

    print(f"\n--- {location} ---")

    for col in ["TMK", "TNK", "TXK", "RSK"]:
        print(
            f"{col}: "
            f"NaN = {df_check[col].isna().sum()}, "
            f"-999 = {df_check[col].eq(-999).sum()}"
        )


--- Dahlem ---
TMK: NaN = 0, -999 = 0
TNK: NaN = 0, -999 = 0
TXK: NaN = 0, -999 = 0
RSK: NaN = 0, -999 = 0

--- Marzahn ---
TMK: NaN = 0, -999 = 7
TNK: NaN = 0, -999 = 7
TXK: NaN = 0, -999 = 7
RSK: NaN = 0, -999 = 7

--- Tempelhof ---
TMK: NaN = 0, -999 = 0
TNK: NaN = 0, -999 = 1
TXK: NaN = 0, -999 = 1
RSK: NaN = 0, -999 = 1


In [93]:
import numpy as np
import pandas as pd

# Match the datasets by location
weather_pairs = {
    "Dahlem": (df_2, df_dwd_3),
    "Marzahn": (df_3, df_dwd_4),
    "Tempelhof": (df_5, df_dwd_5)
}

# Define corresponding weather variables
variable_pairs = {
    "Mean temperature": ("tavg", "TMK"),
    "Minimum temperature": ("tmin", "TNK"),
    "Maximum temperature": ("tmax", "TXK"),
    "Precipitation": ("prcp", "RSK")
}

comparison_results = []

# Compare each location and variable
for location, (df_meteostat, df_dwd) in weather_pairs.items():

    # Prepare temporary copies
    meteostat = df_meteostat.copy()
    dwd = df_dwd.copy()

    # Standardise DWD column names
    dwd.columns = dwd.columns.str.strip()

    # Standardise dates
    meteostat["date"] = pd.to_datetime(
        meteostat["date"]
    ).dt.normalize()

    dwd["date"] = pd.to_datetime(
        dwd["MESS_DATUM"].astype(str).str.strip(),
        format="%Y%m%d"
    )

    # Keep only the relevant DWD variables
    dwd = dwd[
        ["date", "TMK", "TNK", "TXK", "RSK"]
    ]

    # Match observations by date
    merged = pd.merge(
        meteostat,
        dwd,
        on="date",
        how="inner"
    )

    # Compare each variable
    for variable, (met_col, dwd_col) in variable_pairs.items():

        # Keep only dates with values in both sources
        valid = merged[[met_col, dwd_col]].dropna()

        # Calculate absolute differences
        differences = (valid[met_col] - valid[dwd_col]).abs()

        # Calculate comparison statistics
        comparison_results.append({
            "location": location,
            "variable": variable,
            "matched_dates": len(merged),
            "valid_pairs": len(valid),
            "exact_matches": differences.eq(0).sum(),
            "exact_match_pct": differences.eq(0).mean() * 100
                if len(valid) > 0 else np.nan,
            "mean_abs_difference": differences.mean(),
            "max_abs_difference": differences.max(),
            "correlation": valid[met_col].corr(valid[dwd_col])
                if len(valid) > 1 else np.nan
        })

# Create a summary table
df_weather_comparison = pd.DataFrame(comparison_results)

display(
    df_weather_comparison.round(3)
)

,location,variable,matched_dates,valid_pairs,exact_matches,exact_match_pct,mean_abs_difference,max_abs_difference,correlation
0,Dahlem,Mean temperature,283,283,283,100.000,0.000,0.0,1.000
1,Dahlem,Minimum temperature,283,283,283,100.000,0.000,0.0,1.000
2,Dahlem,Maximum temperature,283,283,283,100.000,0.000,0.0,1.000
3,Dahlem,Precipitation,283,283,283,100.000,0.000,0.0,1.000
4,Marzahn,Mean temperature,283,283,283,100.000,0.000,0.0,1.000
5,Marzahn,Minimum temperature,283,283,283,100.000,0.000,0.0,1.000
6,Marzahn,Maximum temperature,283,283,283,100.000,0.000,0.0,1.000
7,Marzahn,Precipitation,283,283,283,100.000,0.000,0.0,1.000
8,Tempelhof,Mean temperature,283,283,283,100.000,0.000,0.0,1.000
9,Tempelhof,Minimum temperature,283,283,282,99.647,3.560,1007.5,0.114


In [94]:
# Prepare temporary copies of the Tempelhof datasets
met = df_5.copy()
dwd = df_dwd_5.copy()

# Standardise DWD column names
dwd.columns = dwd.columns.str.strip()

# Convert dates to a common format
met["date"] = pd.to_datetime(met["date"]).dt.normalize()

dwd["date"] = pd.to_datetime(
    dwd["MESS_DATUM"].astype(str).str.strip(),
    format="%Y%m%d"
)

# Keep the four comparable DWD variables
dwd = dwd[["date", "TMK", "TNK", "TXK", "RSK"]]

# Match both sources by date
tempelhof_check = pd.merge(
    met,
    dwd,
    on="date",
    how="inner"
)

# Identify dates where at least one variable differs
for met_col, dwd_col in [
    ("tavg", "TMK"),
    ("tmin", "TNK"),
    ("tmax", "TXK"),
    ("prcp", "RSK")
]:
    differences = tempelhof_check[
        tempelhof_check[met_col] != tempelhof_check[dwd_col]
    ]

    if not differences.empty:
        print(f"\n--- {met_col} vs {dwd_col} ---")

        display(
            differences[
                ["date", met_col, dwd_col]
            ]
        )


--- tmin vs TNK ---


,date,tmin,TNK
279,2026-10-07,8.5,-999.0



--- tmax vs TXK ---


,date,tmax,TXK
279,2026-10-07,22.4,-999.0



--- prcp vs RSK ---


,date,prcp,RSK
258,2026-09-16,3.7,-999.0


given the above results, the data contained in Meteostats and DWS for Marzahn, Dahlem and Tempelhof

In [ ]:
#adding the identifier column to each dataset
df_weather["source"] = "Meteostat"
df_dwd_weather["source"] = "DWD"

In [98]:
#renaming the columns on the dwd dataset
df_dwd_weather = df_dwd_weather.rename(columns={
    "daily_temp_mean": "tavg",
    "daily_min_temp": "tmin",
    "daily_max_temp": "tmax",
    "precipitation_mm": "prcp"
})

In [109]:
# Rename the DWD station column
df_dwd_weather = df_dwd_weather.rename(columns={
    "station_name": "station"
})

In [110]:
# Combine both weather sources
df_weather_combined = pd.concat(
    [df_meteostat, df_dwd_weather],
    ignore_index=True,
    sort=False
)

# Check the combined dataset
display(df_weather_combined.head())

# Count observations by source and station
display(
    df_weather_combined.groupby(
        ["source", "station"]
    ).size().reset_index(name="observations")
)

,date,tavg,tmin,tmax,prcp,snow,wdir,wspd,wpgt,pres,tsun,station,source,station_id,cloud_cover_mean,humidity_level_mean
0,2026-01-01 00:00:00,2.5,1.8,3.1,3.6,NaN,NaN,23.9,53.7,998.6,0.0,Alexanderplatz,Meteostat,NaN,NaN,NaN
1,2026-01-02 00:00:00,2.2,0.7,2.8,1.7,NaN,NaN,25.2,57.4,988.9,3.0,Alexanderplatz,Meteostat,NaN,NaN,NaN
2,2026-01-03 00:00:00,0.0,-0.6,0.5,3.5,NaN,NaN,19.0,48.2,996.3,23.0,Alexanderplatz,Meteostat,NaN,NaN,NaN
3,2026-01-04 00:00:00,-1.3,-2.8,-0.4,1.0,NaN,NaN,16.8,53.7,1004.3,109.0,Alexanderplatz,Meteostat,NaN,NaN,NaN
4,2026-01-05 00:00:00,-2.0,-4.7,-0.7,0.2,NaN,NaN,11.9,25.9,1012.1,84.0,Alexanderplatz,Meteostat,NaN,NaN,NaN


,source,station,observations
0,DWD,Berlin Brandenburg,550
1,DWD,Buch,550
2,DWD,Dahlem,550
3,DWD,Marzahn,550
4,DWD,Tempelhof,550
5,Meteostat,Alexanderplatz,303
6,Meteostat,Dahlem,303
7,Meteostat,Marzahn,303
8,Meteostat,Tegel,303
9,Meteostat,Tempelhof,303


In [111]:
# Check the dimensions
print("Shape:", df_weather_combined.shape)

# Check source distribution
print("\nObservations by source:")
print(df_weather_combined["source"].value_counts())

# Check observations by station and source
print("\nObservations by station and source:")
display(
    df_weather_combined.groupby(
        ["source", "station"]
    ).size().reset_index(name="observations")
)

# Check missing values
print("\nMissing values:")
display(
    df_weather_combined.isna().sum().to_frame("missing_values")
)

# Check date range by source
print("\nDate coverage:")
display(
    df_weather_combined.groupby("source")["date"].agg(
        ["min", "max", "count"]
    )
)

Shape: (4265, 16)

Observations by source:
source
DWD          2750
Meteostat    1515
Name: count, dtype: int64

Observations by station and source:


,source,station,observations
0,DWD,Berlin Brandenburg,550
1,DWD,Buch,550
2,DWD,Dahlem,550
3,DWD,Marzahn,550
4,DWD,Tempelhof,550
5,Meteostat,Alexanderplatz,303
6,Meteostat,Dahlem,303
7,Meteostat,Marzahn,303
8,Meteostat,Tegel,303
9,Meteostat,Tempelhof,303



Missing values:


,missing_values
date,0
tavg,15
tmin,16
tmax,16
prcp,16
snow,3673
wdir,4265
wspd,2760
wpgt,3053
pres,2760



Date coverage:


,min,max,count
source,,,
DWD,2025-04-09 00:00:00,2026-10-10 00:00:00,2750
Meteostat,2026-01-01 00:00:00,2026-10-30 00:00:00,1515


In [112]:
# Check missing values by source and station
missing_by_station = (
    df_weather_combined
    .groupby(["source", "station"])[
        ["tavg", "tmin", "tmax", "prcp"]
    ]
    .apply(lambda group: group.isna().sum())
    .reset_index()
)

display(missing_by_station)

,source,station,tavg,tmin,tmax,prcp
0,DWD,Berlin Brandenburg,0,0,0,0
1,DWD,Buch,0,0,0,0
2,DWD,Dahlem,0,0,0,0
3,DWD,Marzahn,7,7,7,7
4,DWD,Tempelhof,0,1,1,1
5,Meteostat,Alexanderplatz,0,0,0,0
6,Meteostat,Dahlem,0,0,0,0
7,Meteostat,Marzahn,8,8,8,8
8,Meteostat,Tegel,0,0,0,0
9,Meteostat,Tempelhof,0,0,0,0


In [113]:
# Check whether the DWD missing-value code -999 is still present
core_weather = ["tavg", "tmin", "tmax", "prcp"]

for column in core_weather:
    print(
        column,
        (df_weather_combined[column] == -999).sum()
    )

tavg 0
tmin 0
tmax 0
prcp 0


In [114]:
# Identify dates with missing values in the core weather variables
missing_dates = (
    df_weather_combined[
        df_weather_combined[core_weather].isna().any(axis=1)
    ][["source", "station", "date"] + core_weather]
    .sort_values(["station", "date", "source"])
)

display(missing_dates)

,source,station,date,tavg,tmin,tmax,prcp
3179,DWD,Marzahn,2025-04-23 00:00:00,NaN,NaN,NaN,NaN
3180,DWD,Marzahn,2025-04-24 00:00:00,NaN,NaN,NaN,NaN
3181,DWD,Marzahn,2025-04-25 00:00:00,NaN,NaN,NaN,NaN
3182,DWD,Marzahn,2025-04-26 00:00:00,NaN,NaN,NaN,NaN
3183,DWD,Marzahn,2025-04-27 00:00:00,NaN,NaN,NaN,NaN
3184,DWD,Marzahn,2025-04-28 00:00:00,NaN,NaN,NaN,NaN
3185,DWD,Marzahn,2025-04-29 00:00:00,NaN,NaN,NaN,NaN
901,Meteostat,Marzahn,2026-10-23 00:00:00,NaN,NaN,NaN,NaN
902,Meteostat,Marzahn,2026-10-24 00:00:00,NaN,NaN,NaN,NaN
903,Meteostat,Marzahn,2026-10-25 00:00:00,NaN,NaN,NaN,NaN


In [ ]:
# Calculate the number of valid observations for each core variable
coverage = (
    df_weather_combined
    .groupby(["source", "station"])[["tavg", "tmin", "tmax", "prcp"]]
    .count()
    .reset_index()
)

# Calculate the total number of records per station
total_records = (
    df_weather_combined
    .groupby(["source", "station"])
    .size()
    .reset_index(name="total_records")
)

# Combine valid counts with total records
coverage = coverage.merge(
    total_records,
    on=["source", "station"]
)

# Calculate percentage of valid observations
for column in ["tavg", "tmin", "tmax", "prcp"]:
    coverage[f"{column}_coverage_pct"] = (
        coverage[column] / coverage["total_records"] * 100
    )

display(coverage)

In [116]:
# Check for remaining -999 values in the core weather variables
core_weather = ["tavg", "tmin", "tmax", "prcp"]

for column in core_weather:
    print(
        f"{column}:",
        (df_weather_combined[column] == -999).sum()
    )

tavg: 0
tmin: 0
tmax: 0
prcp: 0


In [117]:
df_weather_combined.to_csv("berlin_weather_combined_all.csv", index=False)